In [1]:
# FAG_Select_Graves_Picture.py
# 12/4/2025


# fileInput="FAG_Relationship_Names_New"
# fileOutput="FAG_Relationship_Names_New_Graves"

# fileIn=DirMain+"Analysis\\"+fileInput+".csv"
# fileOut=DirMain+"OutputData\\ABMC\\"+fileOutput+"_"+timestamp+".csv"

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 

def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])


def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print('\n')
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    for iTry in range(0,2):
        response = requests.request("GET", url, headers=my_header) 
        status_code=response.status_code
        url_source = response.text 
        if status_code!=200:
            print('\tstatus_code:',status_code, 'url',url)
            time.sleep(60)
            continue
        beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
        specific_tag_text = beautifulSoupText.get_text()
        capture_json=json.loads(specific_tag_text) 
        try:
            grave_count=capture_json['total']
        except:
            specific_tag_text
            print(capture_json)
            print(f" XXXX get_grave_count Failed: Try: {iTry} \n url: {url}")
            time.sleep(60) 
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    try:
        collection_df['links']=collection_df['links'].str[0].str[2]
    except:
        collection_df['links']=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df


def next_search_year(search_year,interval,grave_count):
    global interval_max
    next_interval=min(int(interval*10000/grave_count),interval_max)
    first_search_year=search_year-interval
    next_search_year=first_search_year+next_interval
    last_search_year=next_search_year+next_interval
    return next_search_year,first_search_year,last_search_year,next_interval

def interceptor(request):
    request.headers["priority"] = 'u=1, i'
    request.headers["user-agent"] = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36'
    request.headers["cookie"] = '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
    request.headers["sec-ch-ua"] = '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"'
    request.headers["sec-ch-ua-platform"] = '"Windows"'
    request.headers["sec-fetch-mode"] = 'cors'
    
    


blank_line='                                                                                                                                                                                           '


maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract Select Graves'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileInput="Select_Soldier_Names"

fileOutput="Select_Soldier_Names_Graves"


# fileIn=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Inputs\\Select_Names\\"+fileInput+".csv"
fileIn=DirMain+"0-Master Worksheets\\Soldier-Parent Processing\\Inputs\\Select_Names\\"+fileInput+".csv"


# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ Input Data $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
names_df=pd.read_csv(fileIn, encoding='ISO-8859-1')
# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ Input Data $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$



field_list=['lastName','firstName','middleName','fullName','nickName','birthDate','birthMonth','birthYear','deathDate','deathMonth',\
            'deathYear','birthCityName','birthStateName','birthCountryName','cemeteryName','countryName','cemeteryId','links', 'intermentHasPhoto','Parents','Spouses','Siblings','Children','plot']
field_list_set=set(field_list)

url_last_name="https://www.findagrave.com/memorial/search?lastname="
url_first_name="&firstname="
url_death_year ="&deathyear="
url_Suffix="&exactName=true&orderby=d"

## *** Get Listings Data ***




options = webdriver.ChromeOptions()
# chrome_options.add_argument("--headless")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
driver.request_interceptor = interceptor 

for index, row in names_df.iterrows():
    name_index=row['name_index']
    last_name=row['last_name']
    first_name=row['first_name']
    death_year=row['death_year']
    
    url=url_last_name+last_name+url_first_name+first_name+url_death_year+str(death_year)+url_Suffix                   
    print('name_index:',name_index)
    response = driver.get(url) 

    input('')

driver.quit()
print('******** Process Complete *******')

**Starting ***  Extract Select Graves  *** , Time: 2026-01-06 14:22:47.088082  **

name_index: Meyer-Frank-A-19181012


name_index: Meyers-Edward-J-19181026


name_index: Morris-Charles-B-19180927


name_index: Nussberger-George-G-19180927


name_index: Ornsteen-Albert-J-19180928


...

name_index: Spiegel-George-F-19181108


name_index: Steingreber-August-J-19181017


name_index: Timmerman-Herman-T-19181019


name_index: Wald-Marcus-G-19181026


KeyboardInterrupt: Interrupted by user